In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)



def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "'json' or 'python' or 'regex'", 
        "solution_criteria": "The characterstics that the solutions should have to fulfil the task"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [4]:
dataset = generate_dataset()


with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def run_prompt(test_case):
    #combines boths the prompts and the test cases and provide these as messages to claude
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output



In [6]:
#pases in the test cases generated by a model, and the ouptuts to those test cases
def grade_by_model(test_case, output):
       # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    Solution Criteria: {test_case["solution_criteria"]}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [7]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [8]:
def run_test_case(test_case):
    #returns json with the test case, and output from claude with the score
    output = run_prompt(test_case)

    #todo -grading
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    #score = 10

    syntax_score = grade_syntax(output, test_case)
    score = (model_score+syntax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [9]:
def run_eval(test_case):
    results =  []

    total = 0
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

        total+=result["score"]
    avg = total/len(dataset)
    print(avg)


    average = []
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

8.5


In [ ]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket_name(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 object URI (e.g., 's3://my-bucket-name/path/to/file.txt') and extract just the bucket name",
      "format": "regex",
      "solution_criteria": "The regex should match S3 URIs in the format 's3://bucket-name/...' and capture only the bucket name portion. It should handle bucket names with hyphens and numbers, and ignore the path component."
    },
    "score": 8.5,
    "reasoning": "The solution correctly implements the core requirement with a well-formed regex pattern and defensive error handling. The regex `r's3://([^/]+)'` properly matches the protocol, captures the bucket name (using character class `[^/]+` to match anything except forward slash), and ignores the path. However, the implementation lacks robustness features such as input type chec

: 